<a href="https://colab.research.google.com/github/kaziibtisumwasit/parkinsons-disease-detection-voice-ml-thesis/blob/main/Automated_Parkinson%E2%80%99s_Disease_Detection_from_Acoustic_Speech_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# CELL 1 — Install packages required by Colab
import sys
if 'google.colab' in sys.modules:
    !pip install --upgrade pip
    !pip install praat-parselmouth opensmile

In [22]:
# CELL 2 — Mount Drive and define exact paths
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# CELL 3 — Imports
import os
import numpy as np
import pandas as pd
import librosa
import parselmouth
import opensmile
from scipy.stats import skew, kurtosis
from parselmouth.praat import call
import warnings
warnings.filterwarnings("ignore")

In [24]:
# CELL 4 — Helper: statistical summary
def summarize_feature(arr, prefix):
    stats = {}
    arr = arr[np.isfinite(arr)]
    arr = arr[arr > 0] if np.any(arr > 0) else arr

    stats[f"{prefix}_mean"] = np.mean(arr) if arr.size else 0
    stats[f"{prefix}_std"] = np.std(arr) if arr.size else 0
    stats[f"{prefix}_min"] = np.min(arr) if arr.size else 0
    stats[f"{prefix}_max"] = np.max(arr) if arr.size else 0
    stats[f"{prefix}_range"] = stats[f"{prefix}_max"] - stats[f"{prefix}_min"]
    stats[f"{prefix}_skew"] = skew(arr) if arr.size > 2 else 0
    stats[f"{prefix}_kurtosis"] = kurtosis(arr) if arr.size > 3 else 0
    return stats


In [25]:
# CELL 5 — Rhythm features
def calculate_rhythm_features(y, sr):
    features = {}
    frame_length, hop_length = 2048, 512
    RMS_THR, MIN_DUR = 0.25, 0.05

    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    rms = rms / np.max(rms) if np.max(rms) > 1e-6 else np.zeros_like(rms)
    voiced = rms > RMS_THR
    frame_dur = hop_length / sr

    durations, start, active = [], 0, False
    for i in range(len(voiced)):
        if voiced[i] and not active:
            start, active = i, True
        if (not voiced[i] or i == len(voiced) - 1) and active:
            dur = (i - start) * frame_dur
            if dur >= MIN_DUR:
                durations.append(dur)
            active = False

    V = np.array(durations)
    total_time = len(y) / sr

    features["VarcoV"] = np.std(V) / np.mean(V) if V.size and np.mean(V) != 0 else 0
    features["PercentV"] = np.sum(V) / total_time if V.size else 0
    features["nPVI"] = (
        (np.sum(np.abs(np.diff(V))) / np.sum(V[:-1] + V[1:])) *
        (len(V) / (len(V) - 1)) * 100
        if V.size > 1 and np.sum(V[:-1] + V[1:]) != 0 else 0
    )
    return features

In [26]:
# CELL 6 — Main acoustic feature extraction
def extract_all_features(audio_path):
    y, sr = librosa.load(audio_path, sr=None)
    results = {}

    snd = parselmouth.Sound(audio_path)
    pitch = snd.to_pitch()
    harmonicity = snd.to_harmonicity()
    point = call(snd, "To PointProcess (periodic, cc)", 75, 500)

    # -------- JITTER --------
    jp = (0, 0, 0.0001, 0.02, 1.3)
    results["Jitter_local"] = call(point, "Get jitter (local)", *jp)
    results["Jitter_local_abs"] = call(point, "Get jitter (local, absolute)", *jp)
    results["Jitter_rap"] = call(point, "Get jitter (rap)", *jp)
    results["Jitter_ppq5"] = call(point, "Get jitter (ppq5)", *jp)
    results["Jitter_ddp"] = call(point, "Get jitter (ddp)", *jp)

    # -------- SHIMMER --------
    sp = (0, 0, 0.0001, 0.02, 1.3, 1.6)
    results["Shimmer_local"] = call([snd, point], "Get shimmer (local)", *sp)
    results["Shimmer_local_dB"] = call([snd, point], "Get shimmer (local_dB)", *sp)
    results["Shimmer_apq3"] = call([snd, point], "Get shimmer (apq3)", *sp)
    results["Shimmer_apq5"] = call([snd, point], "Get shimmer (apq5)", *sp)
    results["Shimmer_apq11"] = call([snd, point], "Get shimmer (apq11)", *sp)
    results["Shimmer_dda"] = call([snd, point], "Get shimmer (dda)", *sp)

    # -------- HNR --------
    h_vals = harmonicity.values[harmonicity.values > -200]
    results.update(summarize_feature(h_vals, "HNR"))

    # -------- F0 --------
    f0 = pitch.selected_array['frequency']
    f0 = f0[f0 > 0]
    results.update(summarize_feature(f0, "F0"))
    results["F0_slope"] = np.polyfit(np.arange(len(f0)), f0, 1)[0] if len(f0) > 1 else 0

    # -------- FORMANTS --------
    formant = snd.to_formant_burg()
    duration = snd.get_total_duration()
    t_vals = np.linspace(0, duration, max(int(duration * 100), 1))

    for i in range(1, 4):
        vals = np.array([formant.get_value_at_time(i, t) for t in t_vals])
        vals = vals[np.isfinite(vals) & (vals > 0)]
        results.update(summarize_feature(vals, f"F{i}"))

    # -------- MFCC + DELTAS --------
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for name, arr in [
        ("MFCC", mfcc),
        ("DeltaMFCC", librosa.feature.delta(mfcc)),
        ("Delta2MFCC", librosa.feature.delta(mfcc, order=2)),
        ("Delta3MFCC", librosa.feature.delta(mfcc, order=3)),
    ]:
        for i in range(13):
            results.update(summarize_feature(arr[i], f"{name}{i+1}"))

    # -------- SPECTRAL FEATURES --------
    spectral = {
        "ZCR": librosa.feature.zero_crossing_rate(y)[0],
        "Energy": librosa.feature.rms(y=y)[0],
        "Centroid": librosa.feature.spectral_centroid(y=y, sr=sr)[0],
        "Rolloff": librosa.feature.spectral_rolloff(y=y, sr=sr)[0],
        "Bandwidth": librosa.feature.spectral_bandwidth(y=y, sr=sr)[0],
        "Flatness": librosa.feature.spectral_flatness(y=y)[0],
    }

    for k, v in spectral.items():
        results.update(summarize_feature(v, k))
        results.update(summarize_feature(librosa.feature.delta(v), f"Delta{k}"))
        results.update(summarize_feature(librosa.feature.delta(v, order=2), f"Delta2{k}"))

    # -------- SPECTRAL CONTRAST --------
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        results.update(summarize_feature(contrast[i], f"SpecContrast{i+1}"))

    # -------- CHROMA --------
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(12):
        results.update(summarize_feature(chroma[i], f"Chroma{i+1}"))

    # -------- MEL BANDS --------
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=25)
    for i, val in enumerate(np.mean(mel, axis=1)):
        results[f"MelBand_{i+1}"] = val

    # -------- PITCH CONTOUR --------
    f0c, _, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
    )
    f0c = f0c[~np.isnan(f0c)]
    results.update(summarize_feature(f0c, "PitchContour"))
    results["PitchContour_slope"] = np.polyfit(np.arange(len(f0c)), f0c, 1)[0] if len(f0c) > 1 else 0

    # -------- RHYTHM --------
    results.update(calculate_rhythm_features(y, sr))

    return results

In [27]:
# CELL 7 — openSMILE eGeMAPS
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv01a,
    feature_level=opensmile.FeatureLevel.Functionals
)

def extract_egemaps_features(audio_path):
    return smile.process_file(audio_path).iloc[0].to_dict()

In [ ]:
# ===================== BATCH PROCESSING INTO ONE CSV =====================

# English raw dataset folder
BASE_DIR = "/content/drive/MyDrive/Thesis-Final-2026/raw_voice_data/english"

# All generated files will be saved in outputs
OUTPUT_DIR = "/content/drive/MyDrive/Thesis-Final-2026/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "parkinsons_audio_features_combined.csv"
)

folders = ["HC", "PD"]
label_map = {"HC": 0, "PD": 1}

all_rows = []

for folder in folders:
    data_dir = os.path.join(BASE_DIR, folder)

    print(f"\n--- Processing Folder: {folder} ---")

    if not os.path.exists(data_dir):
        print(f"Skipping {folder}: Directory not found at {data_dir}")
        continue

    for fname in sorted(os.listdir(data_dir)):
        if not fname.lower().endswith(".wav"):
            continue

        file_path = os.path.join(data_dir, fname)
        print(f"Processing: {fname}")

        try:
            custom = extract_all_features(file_path)
            egemaps = extract_egemaps_features(file_path)
            features = {**custom, **egemaps}

            base_name = os.path.splitext(fname)[0]

            # Patient ID
            if "-" in base_name:
                patient_id = base_name.split("-")[0]
            else:
                patient_id = base_name.split("_")[0]

            features["Patient_ID"] = patient_id
            features["Class"] = label_map[folder]
            features["File"] = fname
            features["Folder"] = folder

            all_rows.append(features)

            print(f"✔ Features extracted: {len(features)}")

        except Exception as e:
            print(f"❌ Failed on {fname}: {e}")

# ===================== SAVE FINAL CSV =====================

if len(all_rows) > 0:
    final_df = pd.DataFrame(all_rows)

    final_df.to_csv(
        OUTPUT_CSV,
        index=False
    )

    print(f"\n✅ DONE — Combined CSV saved to:")
    print(OUTPUT_CSV)
    print(f"Shape: {final_df.shape}")

else:
    print("\n❌ No data was processed. CSV not created.")


--- Processing Folder: HC ---
Processing: ID00_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID01_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID03_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID05_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID08_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID09_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID10_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID11_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID12_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID14_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID15_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID19_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID21_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID22_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID23_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID25_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID26_hc_0_0_0.wav
✔ Features extracted: 798
Processing: ID28_

In [8]:
# ============================================================
# PHASE 1: STABLE 5-FOLD RFECV
# CELL 1: DRIVE MOUNT + IMPORTS
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from collections import Counter
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Scikit-learn imports
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# CELL 2: LOAD DATA + PATIENT-LEVEL AGGREGATION
# ============================================================

print("\n===== LOADING DATA =====")

CSV_PATH = "/content/drive/MyDrive/Thesis-Final-2026/outputs/parkinsons_audio_features_combined.csv"

OUT_DIR = "/content/drive/MyDrive/Thesis-Final-2026/outputs"

os.makedirs(OUT_DIR, exist_ok=True)

# Load feature CSV
df = pd.read_csv(CSV_PATH)

# Create Patient_ID if it does not already exist
if "Patient_ID" not in df.columns:
    df["Patient_ID"] = (
        df["File"]
        .astype(str)
        .str.replace(".wav", "", regex=False)
        .str.split("-")
        .str[0]
    )

# Prevent patient-level data leakage
df_patient = df.groupby("Patient_ID").mean(numeric_only=True)

df_patient["Class"] = df.groupby("Patient_ID")["Class"].first()

# Separate features and target
feature_cols = [
    c for c in df_patient.columns
    if c != "Class"
]

X = df_patient[feature_cols].values

y = df_patient["Class"].values.astype(int)

feature_names = np.array(feature_cols)

print("Patient-level dataset shape:", X.shape)
print("Number of patients:", len(df_patient))


===== LOADING DATA =====
Patient-level dataset shape: (37, 794)
Number of patients: 37


In [ ]:
# ============================================================
# CELL 3: DEFINE ALL 9 MODELS
# ============================================================

RANDOM_STATE = 42

MODELS = {

    "LogisticRegression": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                solver="liblinear",
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),
        "importance_getter": "named_steps.model.coef_"
    },

    "LinearSVM": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(
                class_weight="balanced",
                max_iter=20000,
                random_state=RANDOM_STATE
            ))
        ]),
        "importance_getter": "named_steps.model.coef_"
    },

    "SGDClassifier": {
        "estimator": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SGDClassifier(
                loss="log_loss",
                class_weight="balanced",
                max_iter=3000,
                tol=1e-3,
                random_state=RANDOM_STATE
            ))
        ]),
        "importance_getter": "named_steps.model.coef_"
    },

    "DecisionTree": {
        "estimator": DecisionTreeClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "importance_getter": "auto"
    },

    "RandomForest": {
        "estimator": RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "importance_getter": "auto"
    },

    "ExtraTrees": {
        "estimator": ExtraTreesClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "importance_getter": "auto"
    },

    "AdaBoost": {
        "estimator": AdaBoostClassifier(
            n_estimators=200,
            learning_rate=0.5,
            random_state=RANDOM_STATE
        ),
        "importance_getter": "auto"
    },

    "GradientBoosting": {
        "estimator": GradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
        "importance_getter": "auto"
    },

    "XGBoost": {
        "estimator": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "importance_getter": "auto"
    }
}

print("Total models:", len(MODELS))
print("Models:", list(MODELS.keys()))

Total models: 9
Models: ['LogisticRegression', 'LinearSVM', 'SGDClassifier', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'AdaBoost', 'GradientBoosting', 'XGBoost']


In [10]:
# ============================================================
# CELL 4: RUN STABLE 5-FOLD RFECV
# ============================================================

print("\n===== STARTING STABLE 5-FOLD RFECV =====")

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

all_results = []

for model_name, config in MODELS.items():

    print(f"\nRunning {model_name}...")

    fold_accuracies = []
    fold_feature_lists = []
    all_features_selected_in_model = []

    for fold_idx, (train_index, test_index) in enumerate(
        outer_cv.split(X, y)
    ):

        X_train = X[train_index]
        X_test = X[test_index]

        y_train = y[train_index]
        y_test = y[test_index]

        # Stable RFECV:
        # Never drop below 20 features
        selector = RFECV(
            estimator=config["estimator"],
            step=5,
            cv=3,
            scoring="accuracy",
            n_jobs=-1,
            min_features_to_select=20,
            importance_getter=config["importance_getter"]
        )

        # Train RFECV
        selector.fit(X_train, y_train)

        # Evaluate on outer test fold
        acc = selector.score(X_test, y_test)

        fold_accuracies.append(acc)

        # Get selected feature names
        selected_features = list(
            feature_names[selector.support_]
        )

        fold_feature_lists.append(selected_features)

        all_features_selected_in_model.extend(
            selected_features
        )

        print(
            f"  -> Fold {fold_idx + 1}: "
            f"Accuracy = {acc:.4f} | "
            f"Features Kept = {len(selected_features)}"
        )


===== STARTING STABLE 5-FOLD RFECV =====

Running LogisticRegression...
  -> Fold 1: Accuracy = 0.6250 | Features Kept = 44
  -> Fold 2: Accuracy = 0.8750 | Features Kept = 284
  -> Fold 3: Accuracy = 0.8571 | Features Kept = 59
  -> Fold 4: Accuracy = 0.8571 | Features Kept = 20
  -> Fold 5: Accuracy = 0.7143 | Features Kept = 224

Running LinearSVM...
  -> Fold 1: Accuracy = 0.6250 | Features Kept = 34
  -> Fold 2: Accuracy = 0.7500 | Features Kept = 414
  -> Fold 3: Accuracy = 0.8571 | Features Kept = 134
  -> Fold 4: Accuracy = 0.8571 | Features Kept = 54
  -> Fold 5: Accuracy = 0.8571 | Features Kept = 159

Running SGDClassifier...
  -> Fold 1: Accuracy = 0.6250 | Features Kept = 94
  -> Fold 2: Accuracy = 0.7500 | Features Kept = 154
  -> Fold 3: Accuracy = 1.0000 | Features Kept = 604
  -> Fold 4: Accuracy = 0.8571 | Features Kept = 669
  -> Fold 5: Accuracy = 0.7143 | Features Kept = 144

Running DecisionTree...
  -> Fold 1: Accuracy = 0.6250 | Features Kept = 524
  -> Fold 2:

In [13]:
# ============================================================
# CELL 5: FEATURE FREQUENCY + EXPORT PHASE 1 RESULTS
# ============================================================

# Count how many times each feature
# appeared across the 5 folds
feature_counts = Counter(
    all_features_selected_in_model
)

# Sort features by frequency
sorted_features_by_frequency = [
    f"{feat} (Count: {cnt})"
    for feat, cnt in feature_counts.most_common()
]

# Store model results
model_data = {
    "Model": model_name,

    "Average_Accuracy": np.mean(
        fold_accuracies
    ),

    "Fold_1_Acc": fold_accuracies[0],
    "Fold_2_Acc": fold_accuracies[1],
    "Fold_3_Acc": fold_accuracies[2],
    "Fold_4_Acc": fold_accuracies[3],
    "Fold_5_Acc": fold_accuracies[4],

    "Fold_1_Features": ", ".join(
        fold_feature_lists[0]
    ),

    "Fold_2_Features": ", ".join(
        fold_feature_lists[1]
    ),

    "Fold_3_Features": ", ".join(
        fold_feature_lists[2]
    ),

    "Fold_4_Features": ", ".join(
        fold_feature_lists[3]
    ),

    "Fold_5_Features": ", ".join(
        fold_feature_lists[4]
    ),

    "Feature_Frequencies": " | ".join(
        sorted_features_by_frequency
    )
}

all_results.append(model_data)


# ============================================================
# EXPORT FOR MANUAL REVIEW
# ============================================================

results_df = pd.DataFrame(
    all_results
).sort_values(
    by="Average_Accuracy",
    ascending=False
)

output_path = os.path.join(
    OUT_DIR,
    "stable_5fold_feature_selection_data.csv"
)

results_df.to_csv(
    output_path,
    index=False
)

print("\n===== PHASE 1 COMPLETE =====")

display(
    results_df[
        ["Model", "Average_Accuracy"]
    ]
)

print(
    f"Data saved to: {output_path}"
)

print(
    "Open the CSV to review feature stability "
    "across all 5 folds."
)


===== PHASE 1 COMPLETE =====


,Model,Average_Accuracy
0,XGBoost,0.792857


Data saved to: /content/drive/MyDrive/Thesis-Final-2026/outputs/stable_5fold_feature_selection_data.csv
Open the CSV to review feature stability across all 5 folds.


In [14]:
# ============================================================
# PHASE 2
# CELL 6: IMPORTS
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.linear_model import (
    LogisticRegression,
    SGDClassifier
)

from sklearn.svm import LinearSVC

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

In [15]:
# ============================================================
# CELL 7: 28 GOLDEN FEATURES
# ============================================================

GOLDEN_FEATURES = [

    "Delta2MFCC3_max",
    "Delta2MFCC3_range",
    "Delta3MFCC1_std",
    "Delta3MFCC4_std",

    "ZCR_mean",
    "Rolloff_max",

    "Flatness_mean",
    "Flatness_std",
    "DeltaFlatness_mean",
    "DeltaFlatness_std",
    "Delta2Flatness_max",

    "SpecContrast1_mean",
    "SpecContrast1_skew",
    "SpecContrast1_kurtosis",
    "SpecContrast4_kurtosis",

    "Chroma11_mean",
    "Chroma11_min",

    "MelBand_1",

    "PitchContour_mean",

    "F0semitoneFrom27.5Hz_sma3nz_amean",
    "F0semitoneFrom27.5Hz_sma3nz_percentile80.0",

    "F1frequency_sma3nz_amean",
    "F2frequency_sma3nz_amean",

    "slopeV0-500_sma3nz_amean",
    "slopeV0-500_sma3nz_stddevNorm",

    "mfcc3V_sma3nz_stddevNorm",

    "MeanVoicedSegmentLengthSec",
    "StddevVoicedSegmentLengthSec"
]

print(
    f"\n===== EVALUATING "
    f"{len(GOLDEN_FEATURES)} STABLE BIOMARKERS ====="
)


===== EVALUATING 28 STABLE BIOMARKERS =====


In [16]:
# ============================================================
# CELL 8: LOAD DATA + PATIENT-LEVEL AGGREGATION
#        + SELECT 28 FEATURES
# ============================================================

CSV_PATH = (
    "/content/drive/MyDrive/"
    "Thesis-Final-2026/outputs/"
    "parkinsons_audio_features_combined.csv"
)

OUT_DIR = (
    "/content/drive/MyDrive/"
    "Thesis-Final-2026/outputs"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

# Load feature CSV
df = pd.read_csv(CSV_PATH)

# Create Patient_ID if missing
if "Patient_ID" not in df.columns:

    df["Patient_ID"] = (
        df["File"]
        .astype(str)
        .str.replace(
            ".wav",
            "",
            regex=False
        )
        .str.split("-")
        .str[0]
    )

# Prevent patient-level leakage
df_patient = (
    df.groupby("Patient_ID")
    .mean(numeric_only=True)
)

df_patient["Class"] = (
    df.groupby("Patient_ID")["Class"]
    .first()
)

# Select ONLY the 28 Golden Features
X_golden = (
    df_patient[GOLDEN_FEATURES]
    .values
)

y = (
    df_patient["Class"]
    .values
    .astype(int)
)

print(
    "Patient-level data shape:",
    X_golden.shape
)

print(
    "Number of selected features:",
    X_golden.shape[1]
)

print(
    "Number of patients:",
    len(df_patient)
)

Patient-level data shape: (37, 28)
Number of selected features: 28
Number of patients: 37


In [17]:
# ============================================================
# CELL 9: DEFINE PHASE 2 MODELS + CV
# ============================================================

RANDOM_STATE = 42

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

MODELS = {

    "LogisticRegression":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),

    "LinearSVM":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),

    "SGDClassifier":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", SGDClassifier(
                loss="log_loss",
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),

    "DecisionTree":
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),

    "RandomForest":
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

    "ExtraTrees":
        ExtraTreesClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

    "AdaBoost":
        AdaBoostClassifier(
            n_estimators=200,
            learning_rate=0.5,
            random_state=RANDOM_STATE
        ),

    "GradientBoosting":
        GradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
}

print(
    "Total models:",
    len(MODELS)
)

Total models: 9


In [18]:
# ============================================================
# CELL 10: FINAL 5-FOLD CROSS-VALIDATION
# ============================================================

results = []

scoring_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

for name, model in MODELS.items():

    print(
        f"\nEvaluating {name}..."
    )

    cv_scores = cross_validate(
        model,
        X_golden,
        y,
        cv=cv,
        scoring=scoring_metrics,
        n_jobs=-1
    )

    results.append({

        "Model": name,

        "Accuracy": np.mean(
            cv_scores["test_accuracy"]
        ),

        "Precision": np.mean(
            cv_scores["test_precision"]
        ),

        "Recall": np.mean(
            cv_scores["test_recall"]
        ),

        "F1_Score": np.mean(
            cv_scores["test_f1"]
        ),

        "ROC_AUC": np.mean(
            cv_scores["test_roc_auc"]
        )
    })


Evaluating LogisticRegression...

Evaluating LinearSVM...

Evaluating SGDClassifier...

Evaluating DecisionTree...

Evaluating RandomForest...

Evaluating ExtraTrees...

Evaluating AdaBoost...

Evaluating GradientBoosting...

Evaluating XGBoost...


In [20]:
# ============================================================
# CELL 11: FINAL LEADERBOARD + SAVE RESULTS
# ============================================================

final_df = (
    pd.DataFrame(results)
    .sort_values(
        by="Accuracy",
        ascending=False
    )
)

print(
    "\n===== FINAL EVALUATION "
    "LEADERBOARD (28 FEATURES) ====="
)

display(final_df)

# Save final results
output_csv = os.path.join(
    OUT_DIR,
    "final_validation_28_features.csv"
)

final_df.to_csv(
    output_csv,
    index=False
)

print(
    f"\nFinal Results saved safely at: "
    f"{output_csv}"
)


===== FINAL EVALUATION LEADERBOARD (28 FEATURES) =====


,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC
2,SGDClassifier,0.889286,0.920000,0.866667,0.870000,0.930000
4,RandomForest,0.871429,0.750000,0.750000,0.742857,0.930000
0,LogisticRegression,0.864286,0.920000,0.816667,0.841429,0.900000
5,ExtraTrees,0.842857,0.750000,0.683333,0.702857,0.946667
6,AdaBoost,0.839286,0.700000,0.733333,0.702857,0.930000
8,XGBoost,0.839286,0.650000,0.800000,0.714286,0.916667
1,LinearSVM,0.807143,0.803333,0.816667,0.786190,0.860000
3,DecisionTree,0.782143,0.783333,0.750000,0.757619,0.775000
7,GradientBoosting,0.675000,0.633333,0.433333,0.480952,0.717917



Final Results saved safely at: /content/drive/MyDrive/Thesis-Final-2026/outputs/final_validation_28_features.csv
